In [ ]:
{
  "timestamp": "2026-08-05T12:30:15Z",
  "persons": [
    {
      "id": 1,
      "bbox": [100, 100, 200, 200],
      "ppe": {
        "lab_coat": True,
        "gloves": False,
        "goggles": True
      }
    }
  ],
  "hazards": [
    {
      "type": "open_flame",
      "bbox": [100, 100, 200, 200]
    }
  ]
}

{'timestamp': '2026-08-05T12:30:15Z',
 'persons': [{'id': 1,
   'bbox': [100, 100, 200, 200],
   'ppe': {'lab_coat': True, 'gloves': False, 'goggles': True}}],
 'hazards': [{'type': 'open_flame', 'bbox': [100, 100, 200, 200]}]}

In [ ]:
{
  "timestamp": "2026-08-05T12:30:20Z",
  "events": [
    {
      "type": "ppe_violation",
      "person_id": 1,
      "severity": "high",
      "details": "No gloves near open flame for 12 seconds",
      "action": "trigger_alert_and_log"
    }
  ]
}


{'timestamp': '2026-08-05T12:30:20Z',
 'events': [{'type': 'ppe_violation',
   'person_id': 1,
   'severity': 'high',
   'details': 'No gloves near open flame for 12 seconds',
   'action': 'trigger_alert_and_log'}]}

In [ ]:
!pip install ultralytics

import cv2
import torch
from ultralytics import YOLO
import uuid
import time

class PerceptionAgent:
    def __init__(self, model_path="yolov8n.pt"):
        self.model = YOLO(model_path)

    def _generate_msg_id(self):
        return f"msg_{uuid.uuid4().hex[:8]}"

    def preprocess(self, img_path):
        """Validate + load + resize image."""
        img = cv2.imread(img_path)
        if img is None:
            return {
                "schema_version": "1.0",
                "agent": "perception",
                "msg_type": "preprocessing_error",
                "timestamp": time.time(),
                "msg_id": self._generate_msg_id(),
                "input_id": img_path,
                "error": "Corrupt or unreadable image"
            }
        img = cv2.resize(img, (640, 640))
        return img

    def process_image(self, img_path):
        """Full perception pipeline: preprocess → detect → structured scene message."""
        preprocessed = self.preprocess(img_path)

        # If preprocessing failed, return the error message
        if isinstance(preprocessed, dict) and preprocessed["msg_type"] == "preprocessing_error":
            return preprocessed

        img = preprocessed
        results = self.model(img, verbose=False)[0]

        persons = []
        hazards = []

        for box in results.boxes:
            cls = int(box.cls)
            label = results.names[cls]
            conf = float(box.conf)
            x1, y1, x2, y2 = box.xyxy[0].tolist()

            if label == "person":
                persons.append({
                    "id": len(persons) + 1,
                    "bbox": [x1, y1, x2, y2],
                    "ppe": {
                        "lab_coat": False,   # placeholder
                        "gloves": False,     # placeholder
                        "goggles": False     # placeholder
                    },
                    "confidence": conf
                })
            else:
                hazards.append({
                    "type": label,
                    "bbox": [x1, y1, x2, y2],
                    "confidence": conf
                })

        return {
            "schema_version": "1.0",
            "agent": "perception",
            "msg_type": "scene",
            "timestamp": time.time(),
            "msg_id": self._generate_msg_id(),
            "input_id": img_path,
            "detections": {
                "persons": persons,
                "hazards": hazards
            }
        }

In [ ]:
import uuid
import time

class SafetyAgent:
    def __init__(self, rules=None):
        # Simple rule config
        self.rules = rules or {
            "ppe_required_near_hazard": True,
            "hazard_distance_threshold_px": 120
        }

    def _generate_msg_id(self):
        return f"msg_{uuid.uuid4().hex[:8]}"

    def evaluate(self, scene_msg):
        """Reasoning pipeline: consume scene → apply rules → produce decision message."""
        if scene_msg["msg_type"] != "scene":
            return {
                "schema_version": "1.0",
                "agent": "safety",
                "msg_type": "decision",
                "timestamp": time.time(),
                "msg_id": self._generate_msg_id(),
                "input_id": scene_msg["input_id"],
                "events": []
            }

        persons = scene_msg["detections"]["persons"]
        hazards = scene_msg["detections"]["hazards"]

        events = []

        for person in persons:
            px1, py1, px2, py2 = person["bbox"]
            person_center = ((px1 + px2) / 2, (py1 + py2) / 2)

            for hazard in hazards:
                hx1, hy1, hx2, hy2 = hazard["bbox"]
                hazard_center = ((hx1 + hx2) / 2, (hy1 + hy2) / 2)

                dist = ((person_center[0] - hazard_center[0]) ** 2 +
                        (person_center[1] - hazard_center[1]) ** 2) ** 0.5

                if dist < self.rules["hazard_distance_threshold_px"]:
                    events.append({
                        "event_id": f"evt_{uuid.uuid4().hex[:6]}",
                        "type": "ppe_violation",
                        "person_id": person["id"],
                        "severity": "high",
                        "reason": {
                            "missing_items": ["gloves"],  # placeholder
                            "near_hazard": hazard["type"],
                            "distance_px": dist,
                            "rules_triggered": ["RULE_PPE_NEAR_HAZARD"]
                        },
                        "recommended_action": "trigger_alert_and_log"
                    })

        return {
            "schema_version": "1.0",
            "agent": "safety",
            "msg_type": "decision",
            "timestamp": time.time(),
            "msg_id": self._generate_msg_id(),
            "input_id": scene_msg["input_id"],
            "events": events
        }


In [ ]:
import os
import json
import time
import cv2
import uuid
import torch
from ultralytics import YOLO

def save_json(path, data):
    with open(path, "w") as f:
        json.dump(data, f, indent=2)

class PerceptionAgent:
    def __init__(self, model_path="yolov8n.pt"):
        self.model = YOLO(model_path)

    def _generate_msg_id(self):
        return f"msg_{uuid.uuid4().hex[:8]}"

    def preprocess(self, img_path):
        """Validate + load + resize image."""
        img = cv2.imread(img_path)
        if img is None:
            return {
                "schema_version": "1.0",
                "agent": "perception",
                "msg_type": "preprocessing_error",
                "timestamp": time.time(),
                "msg_id": self._generate_msg_id(),
                "input_id": img_path,
                "error": "Corrupt or unreadable image"
            }
        img = cv2.resize(img, (640, 640))
        return img

    def process_image(self, img_path):
        """Full perception pipeline: preprocess → detect → structured scene message."""
        preprocessed = self.preprocess(img_path)

        # If preprocessing failed, return the error message
        if isinstance(preprocessed, dict) and preprocessed["msg_type"] == "preprocessing_error":
            return preprocessed

        img = preprocessed
        results = self.model(img, verbose=False)[0]

        persons = []
        hazards = []

        for box in results.boxes:
            cls = int(box.cls)
            label = results.names[cls]
            conf = float(box.conf)
            x1, y1, x2, y2 = box.xyxy[0].tolist()

            if label == "person":
                persons.append({
                    "id": len(persons) + 1,
                    "bbox": [x1, y1, x2, y2],
                    "ppe": {
                        "lab_coat": False,   # placeholder
                        "gloves": False,     # placeholder
                        "goggles": False     # placeholder
                    },
                    "confidence": conf
                })
            else:
                hazards.append({
                    "type": label,
                    "bbox": [x1, y1, x2, y2],
                    "confidence": conf
                })

        return {
            "schema_version": "1.0",
            "agent": "perception",
            "msg_type": "scene",
            "timestamp": time.time(),
            "msg_id": self._generate_msg_id(),
            "input_id": img_path,
            "detections": {
                "persons": persons,
                "hazards": hazards
            }
        }

class SafetyAgent:
    def __init__(self, rules=None):
        # Simple rule config
        self.rules = rules or {
            "ppe_required_near_hazard": True,
            "hazard_distance_threshold_px": 120
        }

    def _generate_msg_id(self):
        return f"msg_{uuid.uuid4().hex[:8]}"

    def evaluate(self, scene_msg):
        """Reasoning pipeline: consume scene → apply rules → produce decision message."""
        if scene_msg["msg_type"] != "scene":
            return {
                "schema_version": "1.0",
                "agent": "safety",
                "msg_type": "decision",
                "timestamp": time.time(),
                "msg_id": self._generate_msg_id(),
                "input_id": scene_msg["input_id"],
                "events": []
            }

        persons = scene_msg["detections"]["persons"]
        hazards = scene_msg["detections"]["hazards"]

        events = []

        for person in persons:
            px1, py1, px2, py2 = person["bbox"]
            person_center = ((px1 + px2) / 2, (py1 + py2) / 2)

            for hazard in hazards:
                hx1, hy1, hx2, hy2 = hazard["bbox"]
                hazard_center = ((hx1 + hx2) / 2, (hy1 + hy2) / 2)

                dist = ((person_center[0] - hazard_center[0]) ** 2 +
                        (person_center[1] - hazard_center[1]) ** 2) ** 0.5

                if dist < self.rules["hazard_distance_threshold_px"]:
                    events.append({
                        "event_id": f"evt_{uuid.uuid4().hex[:6]}",
                        "type": "ppe_violation",
                        "person_id": person["id"],
                        "severity": "high",
                        "reason": {
                            "missing_items": ["gloves"],  # placeholder
                            "near_hazard": hazard["type"],
                            "distance_px": dist,
                            "rules_triggered": ["RULE_PPE_NEAR_HAZARD"]
                        },
                        "recommended_action": "trigger_alert_and_log"
                    })

        return {
            "schema_version": "1.0",
            "agent": "safety",
            "msg_type": "decision",
            "timestamp": time.time(),
            "msg_id": self._generate_msg_id(),
            "input_id": scene_msg["input_id"],
            "events": events
        }

def annotate_image(img_path, scene_msg, decision_msg, out_path):
    img = cv2.imread(img_path)
    if img is None:
        return

    # Draw persons
    for p in scene_msg["detections"]["persons"]:
        x1, y1, x2, y2 = map(int, p["bbox"])
        cv2.rectangle(img, (x1, y1), (x2, y2), (0,255,0), 2)
        cv2.putText(img, f"Person {p['id']}", (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

    # Draw hazards
    for h in scene_msg["detections"]["hazards"]:
        x1, y1, x2, y2 = map(int, h["bbox"])
        cv2.rectangle(img, (x1, y1), (x2, y2), (0,0,255), 2)
        cv2.putText(img, h["type"], (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)

    cv2.imwrite(out_path, img)

def main():
    # Create agents
    perception = PerceptionAgent()
    safety = SafetyAgent()

    # Create run directory
    run_id = f"run_{time.strftime('%Y%m%d_%H%M%S')}"
    base_dir = f"results/{run_id}"
    os.makedirs(f"{base_dir}/annotated", exist_ok=True)
    os.makedirs(f"{base_dir}/traces", exist_ok=True)

    # Input ingestion: batch of images
    input_folder = "data/input/images"
    os.makedirs(input_folder, exist_ok=True) # Create the input images directory
    img_paths = [os.path.join(input_folder, f) for f in os.listdir(input_folder)]

    for img_path in img_paths:
        # Perception agent
        scene_msg = perception.process_image(img_path)
        save_json(f"{base_dir}/traces/{os.path.basename(img_path)}_scene.json", scene_msg)

        # Safety agent
        decision_msg = safety.evaluate(scene_msg)
        save_json(f"{base_dir}/traces/{os.path.basename(img_path)}_decision.json", decision_msg)

        # Action: annotate image + save report
        annotated_path = f"{base_dir}/annotated/{os.path.basename(img_path)}"
        annotate_image(img_path, scene_msg, decision_msg, annotated_path)

        action_record = {
            "input_id": img_path,
            "annotated_image": annotated_path,
            "events": decision_msg["events"]
        }
        save_json(f"{base_dir}/traces/{os.path.basename(img_path)}_action.json", action_record)

        print(f"Processed {img_path}")

if __name__ == "__main__":
    main()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!find /content/drive/MyDrive -type d -iname "*ppe*"

/content/drive/MyDrive/ppe
/content/drive/MyDrive/ppe_yolo11


In [ ]:
!find /content/drive/MyDrive -type d -iname "*ppe*" -print0 | xargs -0 -I {} echo "[{}]"

[/content/drive/MyDrive/ppe]
[/content/drive/MyDrive/ppe_yolo11]


In [ ]:
%%writefile /content/drive/MyDrive/ppe/ppe.yaml
train: /content/drive/MyDrive/ppe/train/images
val: /content/drive/MyDrive/ppe/val/images
test: /content/drive/MyDrive/ppe/test/images

nc: 8
names: ["gloves", "goggles", "lab_coat", "mask", "face_shield", "hard_hat", "apron", "safety_vest"]

Overwriting /content/drive/MyDrive/ppe/ppe.yaml


In [ ]:
!ls /content/drive/MyDrive

 20200331_201044.jpg
 20200331_201115.jpg
 20200331_211801.jpg
'20200401_090257 (1).jpg'
 20200401_090257.jpg
'20200401_090313 (1).jpg'
 20200401_090313.jpg
 20200401_091859.mp4
 20200404_105755.jpg
 20200404_111037.jpg
 20200409_100311.jpg
 20200409_100935.jpg
 20200414_172621.jpg
 20200414_172627.jpg
 20200414_172645.jpg
 20200416_114454.jpg
 20200416_114502.jpg
 20200416_114512.jpg
 20200417_150949.jpg
 20200417_151753.jpg
 20200417_153230.jpg
 20200417_154129.jpg
 20200417_154141.jpg
 20200417_154150.jpg
 20200420_190421.jpg
 20200420_190437.jpg
 20200420_190448.jpg
 20200421_133959.jpg
 20200422_102451.jpg
 20200422_102748.jpg
 20200422_103738.jpg
 20200422_103742.jpg
 20200422_105309.jpg
 20200422_105317.jpg
 20200423_095858.jpg
 20200423_095908.jpg
 20200423_100449.jpg
 20200423_100543.jpg
 20200423_100555.jpg
 20200423_101519.jpg
 20200427_162136.jpg
 20200427_162145.jpg
 20200427_162155.jpg
 20200427_162207.jpg
 20200427_162219.jpg
 20200427_162222.jpg
 20200427_162720.jpg
 20

In [ ]:
!ls /content/drive/MyDrive | grep -i ".zip"

In [ ]:
!unzip /content/drive/MyDrive/PPE-Detection-1.zip -d /content/drive/MyDrive/ppe_yolo11

unzip:  cannot find or open /content/drive/MyDrive/PPE-Detection-1.zip, /content/drive/MyDrive/PPE-Detection-1.zip.zip or /content/drive/MyDrive/PPE-Detection-1.zip.ZIP.


In [ ]:
!ls -R /content/drive/MyDrive/ppe_yolo11

/content/drive/MyDrive/ppe_yolo11:


In [ ]:
!ls -l /content/drive/MyDrive/ppe/train/labels
!ls -l /content/drive/MyDrive/ppe/val/labels
!ls -l /content/drive/MyDrive/ppe/test/labels

total 3362
-rw------- 1 root root 124827 Aug  6 23:42  Lab_PPE.txt
-rw------- 1 root root  42436 Aug  8 05:20 'response_0_.output_image (10).txt'
-rw------- 1 root root 420885 Aug  8 05:20 'response_0_.output_image (1).txt'
-rw------- 1 root root 408035 Aug  8 05:12 'response_0_.output_image (2).txt'
-rw------- 1 root root 304508 Aug  8 05:12 'response_0_.output_image (3).txt'
-rw------- 1 root root 303380 Aug  8 05:18 'response_0_.output_image (4).txt'
-rw------- 1 root root 217215 Aug  8 05:18 'response_0_.output_image (5).txt'
-rw------- 1 root root 238833 Aug  8 05:18 'response_0_.output_image (6).txt'
-rw------- 1 root root 990877 Aug  8 05:18 'response_0_.output_image (7).txt'
-rw------- 1 root root  84413 Aug  8 05:19 'response_0_.output_image (8).txt'
-rw------- 1 root root 304848 Aug  8 05:19 'response_0_.output_image (9).txt'
total 3321
-rw------- 1 root root 124827 Aug  6 23:44  Lab_PPE.txt
-rw------- 1 root root 408035 Aug  8 05:00 'response_0_.output_image (2).txt'
-rw----

In [ ]:
import os

label_dirs = [
    "/content/drive/MyDrive/ppe/train/labels",
    "/content/drive/MyDrive/ppe/val/labels",
    "/content/drive/MyDrive/ppe/test/labels"
]

for d in label_dirs:
    for f in os.listdir(d):
        if not f.lower().endswith(".txt"):
            print("Deleting:", os.path.join(d, f))
            os.remove(os.path.join(d, f))

In [ ]:
!head /content/drive/MyDrive/ppe/train/labels/*.txt

==> /content/drive/MyDrive/ppe/train/labels/Lab_PPE.txt <==
�PNG

i�MR!���>����r������}����v
     @Ȁ��     ��N     -@�     Z�:     � u    @h�    ���	     ��     B��v���Uh�V_7�J�alK&f3A@�:v]E���E���Vh�Z��qx��pQJ�����Fw��!���h�S����n�2ر�\&�I�W(��x(c����D�l(�6���ʕ���g���J�v(�Dġ�
�����^�Qi���E�:�J]�jz"TfX_H#S@���*a.z��u��Z����ޭ[��Q]&tr��@�`W\��#ˏ(46r�T~\��;<<)6�Nn�h��W�9��X��*���m�%�[���2�X����#���������r۱j�p�Ǐ�5�?����m*?PMa-8tǼùM�ﱹ�)"S����1�����$	M<��{_�ޑ� ��yٌer+�l���i�èd1�M��fSy~ٖ���ⱏm[���Y	n<�j���6�x�i��k�J�~�j�;חX�m�K�����[�鿅9�W�	   �ɵ^�{�g�x`�e�[C�xp��_�y`QA����v���^�0�O�@ud�S�䗚�$  �+�N�)l����;�2��
�2�
}u��:�I�v	]��PqO~2tHÓhv̎]/W��8M $$E�2��b��Z�֌�S&J��ڨ���cv���͏��5U�K��+M�1;r4�P'W	�ꄚ4}���c_9\~!�������4�A�c��-���k����\�R
��gO�lz���Gt�w��C�h��֞X���X��|��0Al��Ys�v�)�U\٫�����`",�o�Y}X�����L�G4Vr�NS���w�ď��vU�?���x�<����Cc�t�;��ܛ�|���O��ŧ_޷�S��n�Nl'.��ʋ�������3]k���yo��4�Py��7w�P�E+��۶��]

In [ ]:
!rm -f /content/drive/MyDrive/ppe/train/labels.cache
!rm -f /content/drive/MyDrive/ppe/val/labels.cache
!rm -f /content/drive/MyDrive/ppe/test/labels.cache

In [ ]:
import os

label_dirs = [
    "/content/drive/MyDrive/ppe/train/labels",
    "/content/drive/MyDrive/ppe/val/labels",
    "/content/drive/MyDrive/ppe/test/labels"
]

for d in label_dirs:
    for f in os.listdir(d):
        if f.lower().endswith((".png", ".jpg", ".jpeg")):
            print("Deleting:", os.path.join(d, f))
            os.remove(os.path.join(d, f))


In [ ]:
!ls /content/drive/MyDrive/ppe/train/labels

 Lab_PPE.txt			     'response_0_.output_image (5).txt'
'response_0_.output_image (10).txt'  'response_0_.output_image (6).txt'
'response_0_.output_image (1).txt'   'response_0_.output_image (7).txt'
'response_0_.output_image (2).txt'   'response_0_.output_image (8).txt'
'response_0_.output_image (3).txt'   'response_0_.output_image (9).txt'
'response_0_.output_image (4).txt'


In [ ]:
!ls /content/drive/MyDrive/ppe/train/images

 Lab_PPE.png			     'response_0_.output_image (5).jpg'
'response_0_.output_image (10).jpg'  'response_0_.output_image (6).jpg'
'response_0_.output_image (1).jpg'   'response_0_.output_image (7).jpg'
'response_0_.output_image (2).jpg'   'response_0_.output_image (8).jpg'
'response_0_.output_image (3).jpg'   'response_0_.output_image (9).jpg'
'response_0_.output_image (4).jpg'


In [ ]:
!ls -R "/content/drive/MyDrive/ppe"

/content/drive/MyDrive/ppe:
images	labels	ppe.yaml  test	train  val

/content/drive/MyDrive/ppe/images:
test  train  val

/content/drive/MyDrive/ppe/images/test:

/content/drive/MyDrive/ppe/images/train:

/content/drive/MyDrive/ppe/images/val:

/content/drive/MyDrive/ppe/labels:
test  train  val

/content/drive/MyDrive/ppe/labels/test:

/content/drive/MyDrive/ppe/labels/train:

/content/drive/MyDrive/ppe/labels/val:

/content/drive/MyDrive/ppe/test:
images	labels

/content/drive/MyDrive/ppe/test/images:
 Lab_PPE.png			     'response_0_.output_image (5).jpg'
'response_0_.output_image (10).jpg'  'response_0_.output_image (6).jpg'
'response_0_.output_image (1).jpg'   'response_0_.output_image (7).jpg'
'response_0_.output_image (2).jpg'   'response_0_.output_image (8).jpg'
'response_0_.output_image (3).jpg'   'response_0_.output_image (9).jpg'
'response_0_.output_image (4).jpg'

/content/drive/MyDrive/ppe/test/labels:

/content/drive/MyDrive/ppe/train:
images	labels

/content/drive/MyDrive

In [ ]:
%%writefile /content/drive/MyDrive/ppe/ppe.yaml
train: /content/drive/MyDrive/ppe/train/images
val: /content/drive/MyDrive/ppe/val/images
test: /content/drive/MyDrive/ppe/test/images

nc: 8
names:
  - gloves
  - goggles
  - lab_coat
  - mask
  - face_shield
  - hard_hat
  - apron
  - safety_vest

Overwriting /content/drive/MyDrive/ppe/ppe.yaml


In [ ]:
!ls -R "/content/drive/MyDrive/ppe"

/content/drive/MyDrive/ppe:
images	labels	ppe.yaml  test	train  val

/content/drive/MyDrive/ppe/images:
test  train  val

/content/drive/MyDrive/ppe/images/test:

/content/drive/MyDrive/ppe/images/train:

/content/drive/MyDrive/ppe/images/val:

/content/drive/MyDrive/ppe/labels:
test  train  val

/content/drive/MyDrive/ppe/labels/test:

/content/drive/MyDrive/ppe/labels/train:

/content/drive/MyDrive/ppe/labels/val:

/content/drive/MyDrive/ppe/test:
images	labels

/content/drive/MyDrive/ppe/test/images:
 Lab_PPE.png			     'response_0_.output_image (5).jpg'
'response_0_.output_image (10).jpg'  'response_0_.output_image (6).jpg'
'response_0_.output_image (1).jpg'   'response_0_.output_image (7).jpg'
'response_0_.output_image (2).jpg'   'response_0_.output_image (8).jpg'
'response_0_.output_image (3).jpg'   'response_0_.output_image (9).jpg'
'response_0_.output_image (4).jpg'

/content/drive/MyDrive/ppe/test/labels:

/content/drive/MyDrive/ppe/train:
images	labels

/content/drive/MyDrive

In [ ]:
!ls -R "/content/drive/MyDrive/ppe"

/content/drive/MyDrive/ppe:
images	labels	ppe.yaml  test	train  val

/content/drive/MyDrive/ppe/images:
test  train  val

/content/drive/MyDrive/ppe/images/test:

/content/drive/MyDrive/ppe/images/train:

/content/drive/MyDrive/ppe/images/val:

/content/drive/MyDrive/ppe/labels:
test  train  val

/content/drive/MyDrive/ppe/labels/test:

/content/drive/MyDrive/ppe/labels/train:

/content/drive/MyDrive/ppe/labels/val:

/content/drive/MyDrive/ppe/test:
images	labels

/content/drive/MyDrive/ppe/test/images:
 Lab_PPE.png			     'response_0_.output_image (5).jpg'
'response_0_.output_image (10).jpg'  'response_0_.output_image (6).jpg'
'response_0_.output_image (1).jpg'   'response_0_.output_image (7).jpg'
'response_0_.output_image (2).jpg'   'response_0_.output_image (8).jpg'
'response_0_.output_image (3).jpg'   'response_0_.output_image (9).jpg'
'response_0_.output_image (4).jpg'

/content/drive/MyDrive/ppe/test/labels:

/content/drive/MyDrive/ppe/train:
images	labels

/content/drive/MyDrive

In [ ]:
%%writefile /content/drive/MyDrive/ppe/ppe.yaml
train: /content/drive/MyDrive/ppe/train/images
val: /content/drive/MyDrive/ppe/valid/images
test: /content/drive/MyDrive/ppe/test/images

nc: 8
names: ["gloves", "goggles", "lab_coat", "mask", "face_shield", "hard_hat", "apron", "safety_vest"]

Overwriting /content/drive/MyDrive/ppe/ppe.yaml


In [ ]:
from google.colab import files

In [ ]:
import cv2

# Placeholder classes for demonstration
class PerceptionAgent:
    def process_frame(self, frame):
        print(f"PerceptionAgent processing frame: {frame}")
        return "scene_message_placeholder"

class SafetyAgent:
    def evaluate_scene(self, scene_msg):
        print(f"SafetyAgent evaluating scene: {scene_msg}")
        return "decision_message_placeholder"

class ActionExecutor:
    def execute(self, decision_msg):
        print(f"ActionExecutor executing: {decision_msg}")

# Instantiate the agents
perception_agent = PerceptionAgent()
safety_agent = SafetyAgent()
action_executor = ActionExecutor()

while True:
    cap = cv2.VideoCapture(0)
    video_source = cap
    # The read() method returns a tuple: (boolean, frame)
    # We typically check the boolean to ensure a frame was read successfully
    ret, frame = video_source.read()

    if not ret:
        print("Failed to grab frame")
        break # Exit loop if no frame is captured

    scene_msg = perception_agent.process_frame(frame)
    decision_msg = safety_agent.evaluate_scene(scene_msg)
    action_executor.execute(decision_msg)

    # Release the VideoCapture object after processing each frame in the loop
    # This might not be the most efficient way if you intend to stream continuously
    # but it addresses a potential resource leak if cap is opened repeatedly.
    video_source.release()


Failed to grab frame


In [ ]:
import cv2
import os

input_folder = "data/input/images"
img_paths = [os.path.join(input_folder, f) for f in os.listdir(input_folder)]

for img_path in img_paths:
    frame = cv2.imread(img_path)
    scene_msg = perception_agent.process_frame(frame)
    decision_msg = safety_agent.evaluate_scene(scene_msg)
    action_executor.execute(decision_msg)


In [ ]:
import os
import json
import time
import cv2
import uuid
import torch
from ultralytics import YOLO


# -----------------------------
# Agent Definitions (incorporated for self-containment)
# -----------------------------
class PerceptionAgent:
    def __init__(self, model_path="yolov8n.pt"):
        self.model = YOLO(model_path)

    def _generate_msg_id(self):
        return f"msg_{uuid.uuid4().hex[:8]}"

    def preprocess(self, img_path):
        """Validate + load + resize image."""
        img = cv2.imread(img_path)
        if img is None:
            return {
                "schema_version": "1.0",
                "agent": "perception",
                "msg_type": "preprocessing_error",
                "timestamp": time.time(),
                "msg_id": self._generate_msg_id(),
                "input_id": img_path,
                "error": "Corrupt or unreadable image"
            }
        img = cv2.resize(img, (640, 640))
        return img

    def process_image(self, img_path):
        """Full perception pipeline: preprocess → detect → structured scene message."""
        preprocessed = self.preprocess(img_path)

        # If preprocessing failed, return the error message
        if isinstance(preprocessed, dict) and preprocessed["msg_type"] == "preprocessing_error":
            return preprocessed

        img = preprocessed
        results = self.model(img, verbose=False)[0]

        persons = []
        hazards = []

        for box in results.boxes:
            cls = int(box.cls)
            label = results.names[cls]
            conf = float(box.conf)
            x1, y1, x2, y2 = box.xyxy[0].tolist()

            if label == "person":
                persons.append({
                    "id": len(persons) + 1,
                    "bbox": [x1, y1, x2, y2],
                    "ppe": {
                        "lab_coat": False,   # placeholder
                        "gloves": False,     # placeholder
                        "goggles": False     # placeholder
                    },
                    "confidence": conf
                })
            else:
                hazards.append({
                    "type": label,
                    "bbox": [x1, y1, x2, y2],
                    "confidence": conf
                })

        return {
            "schema_version": "1.0",
            "agent": "perception",
            "msg_type": "scene",
            "timestamp": time.time(),
            "msg_id": self._generate_msg_id(),
            "input_id": img_path,
            "detections": {
                "persons": persons,
                "hazards": hazards
            }
        }

class SafetyAgent:
    def __init__(self, rules=None):
        # Simple rule config
        self.rules = rules or {
            "ppe_required_near_hazard": True,
            "hazard_distance_threshold_px": 120
        }

    def _generate_msg_id(self):
        return f"msg_{uuid.uuid4().hex[:8]}"

    def evaluate(self, scene_msg):
        """Reasoning pipeline: consume scene → apply rules → produce decision message."""
        if scene_msg["msg_type"] != "scene":
            return {
                "schema_version": "1.0",
                "agent": "safety",
                "msg_type": "decision",
                "timestamp": time.time(),
                "msg_id": self._generate_msg_id(),
                "input_id": scene_msg["input_id"],
                "events": []
            }

        persons = scene_msg["detections"]["persons"]
        hazards = scene_msg["detections"]["hazards"]

        events = []

        for person in persons:
            px1, py1, px2, py2 = person["bbox"]
            person_center = ((px1 + px2) / 2, (py1 + py2) / 2)

            for hazard in hazards:
                hx1, hy1, hx2, hy2 = hazard["bbox"]
                hazard_center = ((hx1 + hx2) / 2, (hy1 + hy2) / 2)

                dist = ((person_center[0] - hazard_center[0]) ** 2 +
                        (person_center[1] - hazard_center[1]) ** 2) ** 0.5

                if dist < self.rules["hazard_distance_threshold_px"]:
                    events.append({
                        "event_id": f"evt_{uuid.uuid4().hex[:6]}",
                        "type": "ppe_violation",
                        "person_id": person["id"],
                        "severity": "high",
                        "reason": {
                            "missing_items": ["gloves"],  # placeholder
                            "near_hazard": hazard["type"],
                            "distance_px": dist,
                            "rules_triggered": ["RULE_PPE_NEAR_HAZARD"]
                        },
                        "recommended_action": "trigger_alert_and_log"
                    })

        return {
            "schema_version": "1.0",
            "agent": "safety",
            "msg_type": "decision",
            "timestamp": time.time(),
            "msg_id": self._generate_msg_id(),
            "input_id": scene_msg["input_id"],
            "events": events
        }


# -----------------------------
# Utility functions
# -----------------------------
def save_json(path, data):
    """Save JSON with indentation."""
    with open(path, "w") as f:
        json.dump(data, f, indent=2)


def ensure_folder(path):
    """Create folder if missing."""
    os.makedirs(path, exist_ok=True)


def annotate_image(img_path, scene_msg, decision_msg, out_path):
    """Draw bounding boxes + labels on the image."""
    img = cv2.imread(img_path)
    if img is None:
        return

    # Persons
    for p in scene_msg["detections"].get("persons", []):
        x1, y1, x2, y2 = map(int, p["bbox"])
        cv2.rectangle(img, (x1, y1), (x2, y2), (0,255,0), 2)
        cv2.putText(img, f"Person {p['id']}", (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

    # Hazards
    for h in scene_msg["detections"].get("hazards", []):
        x1, y1, x2, y2 = map(int, h["bbox"])
        cv2.rectangle(img, (x1, y1), (x2, y2), (0,0,255), 2)
        cv2.putText(img, h["type"], (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)

    cv2.imwrite(out_path, img)


# -----------------------------
# Main pipeline
# -----------------------------
def main():
    print("\n=== Multi-Agent CV Pipeline Starting ===\n")

    # -----------------------------
    # 1. Create run directory
    # -----------------------------
    run_id = f"run_{time.strftime('%Y%m%d_%H%M%S')}"
    base_dir = f"results/{run_id}"

    ensure_folder(base_dir)
    ensure_folder(f"{base_dir}/annotated")
    ensure_folder(f"{base_dir}/traces")

    print(f"Run ID: {run_id}")
    print(f"Results directory created at: {base_dir}\n")

    # -----------------------------
    # 2. Input ingestion
    # -----------------------------
    input_folder = '/content/drive/MyDrive/data/input/images'

    if not os.path.exists(input_folder):
        print(f"ERROR: Input folder '{input_folder}' does not exist.")
        print("Create it and upload images, then re-run.")
        return

    img_paths = [
        os.path.join(input_folder, f)
        for f in os.listdir(input_folder)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]

    if len(img_paths) == 0:
        print("ERROR: No images found in data/input/images/")
        print("Upload .jpg or .png files and re-run.")
        return

    print(f"Found {len(img_paths)} images to process.\n")

    # -----------------------------
    # 3. Create agents
    # -----------------------------
    perception = PerceptionAgent()
    safety = SafetyAgent()

    # -----------------------------
    # 4. Process each image
    # -----------------------------
    processed_count = 0
    error_count = 0
    violations_count = 0

    for img_path in img_paths:
        print(f"Processing: {img_path}")

        # Perception agent
        scene_msg = perception.process_image(img_path)
        save_json(f"{base_dir}/traces/{os.path.basename(img_path)}_scene.json", scene_msg)

        # If preprocessing error
        if scene_msg.get("msg_type") == "preprocessing_error":
            print("  -> Preprocessing error logged.")
            error_count += 1
            continue

        # Safety agent
        decision_msg = safety.evaluate(scene_msg)
        save_json(f"{base_dir}/traces/{os.path.basename(img_path)}_decision.json", decision_msg)

        # Count violations
        violations_count += len(decision_msg.get("events", []))

        # Action: annotate image
        annotated_path = f"{base_dir}/annotated/{os.path.basename(img_path)}"
        annotate_image(img_path, scene_msg, decision_msg, annotated_path)

        # Action trace
        action_record = {
            "input_id": img_path,
            "annotated_image": annotated_path,
            "events": decision_msg["events"]
        }
        save_json(f"{base_dir}/traces/{os.path.basename(img_path)}_action.json", action_record)

        processed_count += 1
        print("  -> Done.\n")

    # -----------------------------
    # 5. Clean run summary
    # -----------------------------
    summary = {
        "run_id": run_id,
        "total_inputs": len(img_paths),
        "processed_successfully": processed_count,
        "preprocessing_errors": error_count,
        "total_violations_detected": violations_count,
        "results_dir": base_dir
    }

    save_json(f"{base_dir}/run_summary.json", summary)

    print("=== Run Summary ===")
    print(f"Total images: {summary['total_inputs']}")
    print(f"Processed successfully: {summary['processed_successfully']}")
    print(f"Preprocessing errors: {summary['preprocessing_errors']}")
    print(f"Violations detected: {summary['total_violations_detected']}")
    print(f"Results saved in: {summary['results_dir']}")
    print("\n=== Pipeline Complete ===\n")


if __name__ == "__main__":
    main()


=== Multi-Agent CV Pipeline Starting ===

Run ID: run_20260809_130257
Results directory created at: results/run_20260809_130257

Found 10 images to process.

Processing: /content/drive/MyDrive/data/input/images/response_0_.output_image (1).jpg
  -> Done.

Processing: /content/drive/MyDrive/data/input/images/response_0_.output_image (5).jpg
  -> Done.

Processing: /content/drive/MyDrive/data/input/images/response_0_.output_image (6).jpg
  -> Done.

Processing: /content/drive/MyDrive/data/input/images/response_0_.output_image (7).jpg
  -> Done.

Processing: /content/drive/MyDrive/data/input/images/response_0_.output_image (8).jpg
  -> Done.

Processing: /content/drive/MyDrive/data/input/images/response_0_.output_image (9).jpg
  -> Done.

Processing: /content/drive/MyDrive/data/input/images/response_0_.output_image (10).jpg
  -> Done.

Processing: /content/drive/MyDrive/data/input/images/response_0_.output_image (2).jpg
  -> Done.

Processing: /content/drive/MyDrive/data/input/images/res

In [ ]:
import os
import shutil

# Define the source and destination directories
source_dir = '/content/'
dest_dir = '/content/drive/MyDrive/data/input/images'

# Create the destination directory if it doesn't exist
os.makedirs(dest_dir, exist_ok=True)

print(f"Copying supported image files from {source_dir} to {dest_dir}...")

# List all files in the source directory
for filename in os.listdir(source_dir):
    # Construct full file paths
    source_path = os.path.join(source_dir, filename)
    dest_path = os.path.join(dest_dir, filename)

    # Check if it's a file and a supported image type
    if os.path.isfile(source_path) and filename.lower().endswith(('.jpg', '.jpeg', '.png')):
        try:
            shutil.copy2(source_path, dest_path) # Use copy2 instead of move
            print(f"  Copied: {filename}")
        except Exception as e:
            print(f"  Error copying {filename}: {e}")
    elif os.path.isfile(source_path) and filename.lower().endswith(('.avif')):
        print(f"  Skipped (unsupported AVIF format): {filename}")

print("Image file transfer complete.")

Copying supported image files from /content/ to /content/drive/MyDrive/data/input/images...
Image file transfer complete.


In [ ]:
import json

json_data_string = """
{
  "schema_version": "1.0",
  "agent": "perception",
  "timestamp": "2026-08-05T12:34:15Z",
  "detections": {
    "persons": [
      {
        "id": 3,
        "bbox": [120, 88, 260, 400],
        "ppe": {
          "lab_coat": true,
          "gloves": false,
          "goggles": true
        }
      }
    ],
    "hazards": [
      {
        "type": "open_flame",
        "bbox": [500, 300, 560, 360]
      }
    ]
  }
}
"""

# Parse the JSON string into a Python dictionary
json_data = json.loads(json_data_string)

# You can now work with json_data as a Python dictionary
# For example, to print it:
# print(json_data)

In [ ]:
{
  "schema_version": "1.0",
  "agent": "safety",
  "timestamp": "2026-08-05T12:34:16Z",
  "events": [
    {
      "type": "ppe_violation",
      "person_id": 3,
      "severity": "high",
      "details": "Missing gloves near open flame for 8 seconds",
      "action": "trigger_alert_and_log"
    }
  ]
}


{'schema_version': '1.0',
 'agent': 'safety',
 'timestamp': '2026-08-05T12:34:16Z',
 'events': [{'type': 'ppe_violation',
   'person_id': 3,
   'severity': 'high',
   'details': 'Missing gloves near open flame for 8 seconds',
   'action': 'trigger_alert_and_log'}]}

In [ ]:
class PerceptionAgent:
    def process_frame(self, frame):
        detections = self.model(frame)
        return build_scene_message(detections)

In [ ]:
class SafetyAgent:
    def evaluate(self, scene_msg):
        events = self.rule_engine.check(scene_msg)
        return build_decision_message(events)

In [ ]:
{
  "schema_version": "1.0",
  "agent": "perception",
  "msg_type": "preprocessing_error",
  "timestamp": "2026-08-05T12:40:01Z",
  "input_id": "img_0007.jpg",
  "error": "Corrupt image: cv2.imread returned None"
}


{'schema_version': '1.0',
 'agent': 'perception',
 'msg_type': 'preprocessing_error',
 'timestamp': '2026-08-05T12:40:01Z',
 'input_id': 'img_0007.jpg',
 'error': 'Corrupt image: cv2.imread returned None'}

In [ ]:
import json
json_data_string = """
{
  "schema_version": "1.0",
  "agent": "perception",
  "msg_type": "scene",
  "timestamp": "2026-08-05T12:40:05Z",
  "input_id": "frame_00123",
  "detections": {
    "persons": [
      {
        "id": 1,
        "bbox": [120, 88, 260, 400],
        "ppe": {
          "lab_coat": true,
          "gloves": false,
          "goggles": true
        },
        "confidence": 0.94
      }
    ],
    "hazards": [
      {
        "type": "open_flame",
        "bbox": [500, 300, 560, 360],
        "confidence": 0.91
      }
    ]
  }
}"""


In [ ]:
{
  "schema_version": "1.0",
  "agent": "safety",
  "msg_type": "decision",
  "timestamp": "2026-08-05T12:40:06Z",
  "input_id": "frame_00123",
  "events": [
    {
      "event_id": "evt_0001",
      "type": "ppe_violation",
      "person_id": 1,
      "severity": "high",
      "reason": {
        "missing_items": ["gloves"],
        "near_hazard": "open_flame",
        "distance_px": 75,
        "duration_seconds": 9.2,
        "rules_triggered": ["RULE_PPE_NEAR_FLAME"]
      },
      "recommended_action": "trigger_alert_and_log"
    }
  ]
}


{'schema_version': '1.0',
 'agent': 'safety',
 'msg_type': 'decision',
 'timestamp': '2026-08-05T12:40:06Z',
 'input_id': 'frame_00123',
 'events': [{'event_id': 'evt_0001',
   'type': 'ppe_violation',
   'person_id': 1,
   'severity': 'high',
   'reason': {'missing_items': ['gloves'],
    'near_hazard': 'open_flame',
    'distance_px': 75,
    'duration_seconds': 9.2,
    'rules_triggered': ['RULE_PPE_NEAR_FLAME']},
   'recommended_action': 'trigger_alert_and_log'}]}

In [ ]:
{
  "schema_version": "1.0",
  "agent": "orchestrator",
  "msg_type": "action",
  "timestamp": "2026-08-05T12:40:07Z",
  "input_id": "frame_00123",
  "actions": [
    {
      "event_id": "evt_0001",
      "performed": ["annotate_image", "append_to_run_report"],
      "outputs": {
        "annotated_image_path": "results/run_20260805_1240/annotated/frame_00123.png",
        "report_path": "results/run_20260805_1240/report.json"
      }
    }
  ]
}


{'schema_version': '1.0',
 'agent': 'orchestrator',
 'msg_type': 'action',
 'timestamp': '2026-08-05T12:40:07Z',
 'input_id': 'frame_00123',
 'actions': [{'event_id': 'evt_0001',
   'performed': ['annotate_image', 'append_to_run_report'],
   'outputs': {'annotated_image_path': 'results/run_20260805_1240/annotated/frame_00123.png',
    'report_path': 'results/run_20260805_1240/report.json'}}]}

In [ ]:
{
  "run_id": "run_20260805_1240",
  "input": {
    "input_id": "frame_00123",
    "source": "video:lab_feed.mp4",
    "path": "data/input/video/lab_feed.mp4"
  },
  "perception_message": '{ ...scene message as above... }',
  "decision_message": '{ ...decision message as above... }',
  "action_message": '{ ...action record as above... }'
}


{'run_id': 'run_20260805_1240',
 'input': {'input_id': 'frame_00123',
  'source': 'video:lab_feed.mp4',
  'path': 'data/input/video/lab_feed.mp4'},
 'perception_message': '{ ...scene message as above... }',
 'decision_message': '{ ...decision message as above... }',
 'action_message': '{ ...action record as above... }'}

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO
import uuid
import time

# Re-define or ensure PerceptionAgent is the one with 'model' attribute
# This assumes the full PerceptionAgent class (e.g., from OqJZA9ARFwSR or m6xvAWrxGM8d)
# has been executed and is available in the kernel's scope.
class PerceptionAgent:
    def __init__(self, model_path="yolov8n.pt"):
        self.model = YOLO(model_path)

    def _generate_msg_id(self):
        return f"msg_{uuid.uuid4().hex[:8]}"

    def preprocess(self, img_path):
        img = cv2.imread(img_path)
        if img is None:
            return {
                "schema_version": "1.0",
                "agent": "perception",
                "msg_type": "preprocessing_error",
                "timestamp": time.time(),
                "msg_id": self._generate_msg_id(),
                "input_id": img_path,
                "error": "Corrupt or unreadable image"
            }
        img = cv2.resize(img, (640, 640))
        return img

    def process_image(self, img_path):
        preprocessed = self.preprocess(img_path)
        if isinstance(preprocessed, dict) and preprocessed["msg_type"] == "preprocessing_error":
            return preprocessed
        img = preprocessed
        results = self.model(img, verbose=False)[0]

        persons = []
        hazards = []

        for box in results.boxes:
            cls = int(box.cls)
            label = results.names[cls]
            conf = float(box.conf)
            x1, y1, x2, y2 = box.xyxy[0].tolist()

            if label == "person":
                persons.append({
                    "id": len(persons) + 1,
                    "bbox": [x1, y1, x2, y2],
                    "ppe": {
                        "lab_coat": False,
                        "gloves": False,
                        "goggles": False
                    },
                    "confidence": conf
                })
            else:
                hazards.append({
                    "type": label,
                    "bbox": [x1, y1, x2, y2],
                    "confidence": conf
                })

        return {
            "schema_version": "1.0",
            "agent": "perception",
            "msg_type": "scene",
            "timestamp": time.time(),
            "msg_id": self._generate_msg_id(),
            "input_id": img_path,
            "detections": {
                "persons": persons,
                "hazards": hazards
            }
        }

# Instantiate the correct PerceptionAgent
perception_agent = PerceptionAgent()

# Placeholder for 'img' variable. In a real scenario, 'img' would be a numpy array
# representing an image, often from cv2.imread or a live camera feed.
# Here, we create a dummy image for demonstration purposes.
img = np.zeros((640, 640, 3), dtype=np.uint8) # A black image

self = perception_agent
results = self.model(img, verbose=False)[0]

persons = []
hazards = []

for box in results.boxes:
    cls = int(box.cls)
    label = results.names[cls]
    conf = float(box.conf)
    x1, y1, x2, y2 = box.xyxy[0].tolist()

# Example of how you might continue to populate persons/hazards (based on original snippet's intent)
    if label == "person":
        persons.append({
            "id": len(persons) + 1,
            "bbox": [x1, y1, x2, y2],
            "ppe": {},
            "confidence": conf
        })
    else:
        hazards.append({
            "type": label,
            "bbox": [x1, y1, x2, y2],
            "confidence": conf
        })

print("Processing completed without AttributeError.")
print(f"Detected persons: {len(persons)}")
print(f"Detected hazards: {len(hazards)}")

Processing completed without AttributeError.
Detected persons: 0
Detected hazards: 0


In [ ]:
import numpy as np
from ultralytics import YOLO

# --- Minimal PerceptionAgent definition for 'self.model' to exist ---
# The full PerceptionAgent class is defined elsewhere (e.g., cell m6xvAWrxGM8d).
class PerceptionAgent:
    def __init__(self, model_path="yolov8n.pt"):
        self.model = YOLO(model_path)

# Instantiate the PerceptionAgent
perception_agent = PerceptionAgent()

# Create a dummy 'img' variable (e.g., a black image for demonstration)
# In a real scenario, 'img' would be a preprocessed image (numpy array).
img = np.zeros((640, 640, 3), dtype=np.uint8)

# Assign the PerceptionAgent instance to 'self' for this snippet's context
self = perception_agent

# --- Original code snippet (from PerceptionAgent.process_image) ---
results = self.model(img, verbose=False)[0]

persons = []
hazards = []

# Iterate through detections and print details
if results.boxes:
    for box in results.boxes:
        cls = int(box.cls)
        label = results.names[cls]
        conf = float(box.conf)
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        print(f"Detected: {label} (conf: {conf:.2f}) at bbox: [{x1}, {y1}, {x2}, {y2}]")
        # In the full process_image method, these would be appended to persons/hazards lists
else:
    print("No objects detected in the dummy image.")

print("\nSnippet executed successfully with dummy data.")
print(f"Number of detections (from results): {len(results.boxes)}")


No objects detected in the dummy image.

Snippet executed successfully with dummy data.
Number of detections (from results): 0


In [ ]:
import uuid

# Simulate the SafetyAgent context for this snippet
class MockSafetyAgent:
    def __init__(self):
        self.rules = {
            "hazard_distance_threshold_px": 120 # Example threshold
        }

self = MockSafetyAgent()
events = [] # This list would typically be initialized in the evaluate method

# Example values for person_center and hazard_center
# These would come from bounding box calculations earlier in the evaluate method
person_center = (150, 150) # Example center for a person
hazard_center = (200, 200) # Example center for a hazard
hazard = {"type": "open_flame"} # Example hazard dictionary

dist = ((person_center[0] - hazard_center[0]) ** 2 +
        (person_center[1] - hazard_center[1]) ** 2) ** 0.5

if dist < self.rules["hazard_distance_threshold_px"]:
    events.append({
        "event_id": f"evt_{uuid.uuid4().hex[:6]}", # Added event_id for completeness
        "type": "ppe_violation",
        "reason": {
            "missing_items": ["gloves"],
            "near_hazard": hazard["type"],
            "distance_px": dist,
            "rules_triggered": ["RULE_PPE_NEAR_HAZARD"]
        }
    })

print(f"Distance: {dist}")
print(f"Events generated: {events}")

Distance: 70.71067811865476
Events generated: [{'event_id': 'evt_f8251c', 'type': 'ppe_violation', 'reason': {'missing_items': ['gloves'], 'near_hazard': 'open_flame', 'distance_px': 70.71067811865476, 'rules_triggered': ['RULE_PPE_NEAR_HAZARD']}}]


In [ ]:
import cv2
import os

# --- Mocking context variables for demonstration ---
# In a real pipeline, these would be populated by preceding agent steps.
img_path = "/content/Lab_PPE.png" # Example path, replace with actual if needed for real image processing
scene_msg = {
    "schema_version": "1.0",
    "agent": "perception",
    "msg_type": "scene",
    "timestamp": 1722800000.0,
    "msg_id": "msg_abc12345",
    "input_id": img_path,
    "detections": {
        "persons": [{"id": 1, "bbox": [10,10,50,50]}],
        "hazards": [{"type": "fire", "bbox": [100,100,150,150]}]
    }
}
decision_msg = {
    "schema_version": "1.0",
    "agent": "safety",
    "msg_type": "decision",
    "timestamp": 1722800001.0,
    "msg_id": "msg_def67890",
    "input_id": img_path,
    "events": [
        {"event_id": "evt_001", "type": "warning", "person_id": 1, "severity": "low", "reason": "placeholder"}
    ]
}
annotated_path = "/tmp/annotated_Lab_PPE.png" # Temporary path for saving annotated image

# --- Mocking the annotate_image function if not already defined ---
# In the full pipeline, this function is defined globally.
# For isolated snippet execution, we need to ensure it's present.
def annotate_image(img_path, scene_msg, decision_msg, out_path):
    print(f"Annotating image: {img_path} to {out_path}")
    print(f"Scene: {scene_msg}")
    print(f"Decision: {decision_msg}")
    # Simulate image loading and saving without actual OpenCV operations
    # If cv2 is available and a real image exists, this would do actual annotation.
    try:
        if os.path.exists(img_path):
            img = cv2.imread(img_path)
            if img is not None:
                # Minimal drawing to show it's working
                for p in scene_msg["detections"].get("persons", []):
                    x1, y1, x2, y2 = map(int, p["bbox"])
                    cv2.rectangle(img, (x1, y1), (x2, y2), (0,255,0), 2)
                cv2.imwrite(out_path, img)
                print(f"Dummy annotation saved to {out_path}")
            else:
                print(f"Could not read image at {img_path}. Skipping actual annotation.")
        else:
            print(f"Image file not found at {img_path}. Skipping actual annotation.")
    except Exception as e:
        print(f"Error during dummy annotation: {e}")


# --- Original code snippet ---
annotate_image(img_path, scene_msg, decision_msg, annotated_path)

action_record = {
    "input_id": img_path,
    "annotated_image": annotated_path,
    "events": decision_msg["events"]
}

print("\nAction record generated successfully:")
print(action_record)


Annotating image: /content/Lab_PPE.png to /tmp/annotated_Lab_PPE.png
Scene: {'schema_version': '1.0', 'agent': 'perception', 'msg_type': 'scene', 'timestamp': 1722800000.0, 'msg_id': 'msg_abc12345', 'input_id': '/content/Lab_PPE.png', 'detections': {'persons': [{'id': 1, 'bbox': [10, 10, 50, 50]}], 'hazards': [{'type': 'fire', 'bbox': [100, 100, 150, 150]}]}}
Decision: {'schema_version': '1.0', 'agent': 'safety', 'msg_type': 'decision', 'timestamp': 1722800001.0, 'msg_id': 'msg_def67890', 'input_id': '/content/Lab_PPE.png', 'events': [{'event_id': 'evt_001', 'type': 'warning', 'person_id': 1, 'severity': 'low', 'reason': 'placeholder'}]}
Image file not found at /content/Lab_PPE.png. Skipping actual annotation.

Action record generated successfully:
{'input_id': '/content/Lab_PPE.png', 'annotated_image': '/tmp/annotated_Lab_PPE.png', 'events': [{'event_id': 'evt_001', 'type': 'warning', 'person_id': 1, 'severity': 'low', 'reason': 'placeholder'}]}


In [ ]:
ocr = OCRTool()
result = ocr.extract_text("data/sample/chemical_label.jpg")
print(result)

Performing OCR on: data/sample/chemical_label.jpg
Sample extracted text from chemical label: Glucose, C6H12O6, Purity 99.5%


[12:40:05.001] HANDOFF orchestrator → perception input_id=frame_00123
[12:40:05.210] HANDOFF perception → safety input_id=frame_00123 msg_type=scene
[12:40:06.015] HANDOFF safety → orchestrator input_id=frame_00123 msg_type=decision
[12:40:07.002] ACTION orchestrator input_id=frame_00123 actions=annotate_image,append_to_run_report

[12:34:15.221] FRAME_CAPTURED id=442
[12:34:15.389] HANDOFF perception_agent → safety_agent msg_id=884
[12:34:16.002] HANDOFF safety_agent → orchestrator action=trigger_alert_and_log


Now that your image files are in the correct `data/input/images` directory, you can re-run the `main` function to process them.

In [ ]:
class OCRTool:
    def extract_text(self, image_path):
        # Placeholder for actual OCR logic
        print(f"Performing OCR on: {image_path}")
        # In a real scenario, this would use an OCR library to extract text
        return "Sample extracted text from chemical label: Glucose, C6H12O6, Purity 99.5%"

In [ ]:
import os

# Define the path for the dataset configuration file
dataset_dir = "datasets/ppe"
dataset_yaml_path = os.path.join(dataset_dir, "ppe.yaml")

# Create the directory if it doesn't exist
os.makedirs(dataset_dir, exist_ok=True)

# Create a placeholder ppe.yaml file
# This file typically defines paths to your training/validation images and labels,
# along with the names of your classes.
# You should replace this with your actual dataset configuration.
ppe_yaml_content = """
train: ../train/images
val: ../valid/images

nc: 2  # number of classes
names: ['person', 'ppe_gear']  # class names
"""

with open(dataset_yaml_path, "w") as f:
    f.write(ppe_yaml_content)

print(f"Created placeholder dataset configuration file: {dataset_yaml_path}")
print("Please update this file with your actual dataset paths and class names.")

Created placeholder dataset configuration file: datasets/ppe/ppe.yaml
Please update this file with your actual dataset paths and class names.


In [ ]:
import os

dataset_dir = "datasets/ppe"
dataset_yaml_path = os.path.join(dataset_dir, "ppe.yaml")

# Ensure the directory exists
os.makedirs(dataset_dir, exist_ok=True)

# The content from the user's selected cell, adjusted for correct YAML and YOLO format
ppe_yaml_content = """
path: /content/datasets/ppe # Absolute path to the dataset root
train: images/train        # Path to training images relative to 'path'
val: images/val            # Path to validation images relative to 'path'
test: images/test          # Path to test images relative to 'path'

nc: 5
names: ['lab_coat', 'gloves', 'goggles', 'mask', 'face_shield']
"""

with open(dataset_yaml_path, "w") as f:
    f.write(ppe_yaml_content)

print(f"Updated dataset configuration file: {dataset_yaml_path}")

Updated dataset configuration file: datasets/ppe/ppe.yaml


In [ ]:
import os

# Define the base path for the dataset
base_dataset_path = "/content/datasets/ppe/images"

# Create the required subdirectories for train, val, and test images
os.makedirs(os.path.join(base_dataset_path, "train"), exist_ok=True)
os.makedirs(os.path.join(base_dataset_path, "val"), exist_ok=True)
os.makedirs(os.path.join(base_dataset_path, "test"), exist_ok=True)

print(f"Created placeholder directories for dataset images:")
print(f"- {os.path.join(base_dataset_path, 'train')}")
print(f"- {os.path.join(base_dataset_path, 'val')}")
print(f"- {os.path.join(base_dataset_path, 'test')}")
print("Please populate these directories with your actual images and corresponding label files.")

Created placeholder directories for dataset images:
- /content/datasets/ppe/images/train
- /content/datasets/ppe/images/val
- /content/datasets/ppe/images/test
Please populate these directories with your actual images and corresponding label files.


In [ ]:
pip install -U inference-sdk

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="fo9epdk8tPlgojQcHFxG")
project = rf.workspace("troy-nsofor").project("ppe-detection-dataset")
dataset = project.version(1).download("coco")

loading Roboflow workspace...
loading Roboflow project...


In [ ]:
from inference_sdk import InferenceHTTPClient
import os
import cv2 # Import cv2 for image decoding
import numpy as np # Import numpy for array operations

# 2. Connect to your workflow
client = InferenceHTTPClient(
    api_url="https://serverless.roboflow.com",
    api_key="fo9epdk8tPlgojQcHFxG"
)

# 3. Load the image as bytes
image_path = "/content/drive/MyDrive/ppe/train/images/response_0_.output_image (2).jpg"

# Ensure the file exists before attempting to read it
if not os.path.exists(image_path):
    raise FileNotFoundError(f"Error: Image file not found at {image_path}. Please check the path and try again.")

with open(image_path, "rb") as f:
    image_bytes = f.read()

# Decode image bytes into a NumPy array (e.g., using OpenCV)
# np.frombuffer converts bytes to a 1D array, then cv2.imdecode decodes it into an image
image_np = cv2.imdecode(np.frombuffer(image_bytes, np.uint8), cv2.IMREAD_COLOR)

# Check if image was decoded successfully
if image_np is None:
    raise ValueError(f"Could not decode image from bytes for path: {image_path}")

# 4. Run your workflow on the image (now a NumPy array)
result = client.run_workflow(
    workspace_name="troy-nsofor",
    workflow_id="ppe-detection-dataset-qdp63",
    images={
        "image": image_np # Pass image as a NumPy array
    },
    use_cache=True # Speeds up repeated requests
)

# 5. Get your results
print(result)

[{'predictions': {'image': {'width': None, 'height': None}, 'predictions': []}, 'inference_id': '81c08919-457d-4591-aed0-61446f59ced2', 'model_id': 'ppe-detection-dataset-qdp63/2'}]


In [ ]:
import os

dataset_dir = "datasets/ppe"
dataset_yaml_path = os.path.join(dataset_dir, "ppe.yaml")

# Ensure the directory exists
os.makedirs(dataset_dir, exist_ok=True)

# The following content, which caused a SyntaxError when interpreted directly as Python,
# is now stored as a multi-line string.
ppe_yaml_content = """
path: /content/datasets/ppe # Absolute path to the dataset root
train: images/train        # Path to training images relative to 'path'
val: images/val            # Path to validation images relative to 'path'
test: images/test          # Path to test images relative to 'path'

nc: 5
names: ['lab_coat', 'gloves', 'goggles', 'mask', 'face_shield']
"""

with open(dataset_yaml_path, "w") as f:
    f.write(ppe_yaml_content)

print(f"Dataset configuration file updated: {dataset_yaml_path}")
print("This cell now correctly defines and writes the YOLO dataset YAML configuration.")

Dataset configuration file updated: datasets/ppe/ppe.yaml
This cell now correctly defines and writes the YOLO dataset YAML configuration.


In [ ]:
!ls -R /content/datasets/ppe

/content/datasets/ppe:
images	ppe.yaml

/content/datasets/ppe/images:
test  train  val

/content/datasets/ppe/images/test:

/content/datasets/ppe/images/train:

/content/datasets/ppe/images/val:


### Copying your dataset from Google Drive to Colab

In [127]:
import os
import shutil

# !!! IMPORTANT: Update this path to the root of your dataset in Google Drive !!!
DRIVE_DATASET_PATH = "/content/drive/MyDrive/ppe_yolo11"

# Destination paths in Colab
COLAB_BASE_PATH = '/content/drive/MyDrive/ppe'
COLAB_TRAIN_IMAGES = os.path.join(COLAB_BASE_PATH, 'images/train')
COLAB_VAL_IMAGES = os.path.join(COLAB_BASE_PATH, 'images/val')
COLAB_TEST_IMAGES = os.path.join(COLAB_BASE_PATH, 'images/test')
COLAB_TRAIN_LABELS = os.path.join(COLAB_BASE_PATH, 'labels/train')
COLAB_VAL_LABELS = os.path.join(COLAB_BASE_PATH, 'labels/val')
COLAB_TEST_LABELS = os.path.join(COLAB_BASE_PATH, 'labels/test')

# Ensure destination directories exist
os.makedirs(COLAB_TRAIN_IMAGES, exist_ok=True)
os.makedirs(COLAB_VAL_IMAGES, exist_ok=True)
os.makedirs(COLAB_TEST_IMAGES, exist_ok=True)
os.makedirs(COLAB_TRAIN_LABELS, exist_ok=True)
os.makedirs(COLAB_VAL_LABELS, exist_ok=True)
os.makedirs(COLAB_TEST_LABELS, exist_ok=True)

def copy_files(src_dir, dest_dir, file_type):
    if not os.path.exists(src_dir):
        print(f"Warning: Source directory not found: {src_dir}")
        return
    print(f"Copying {file_type} from {src_dir} to {dest_dir}...")
    for filename in os.listdir(src_dir):
        shutil.copy2(os.path.join(src_dir, filename), os.path.join(dest_dir, filename))
    print(f"Finished copying {file_type}.")

# Copy training data
copy_files(os.path.join(DRIVE_DATASET_PATH, 'train/images'), COLAB_TRAIN_IMAGES, 'training images')
copy_files(os.path.join(DRIVE_DATASET_PATH, 'train/labels'), COLAB_TRAIN_LABELS, 'training labels')

# Copy validation data
copy_files(os.path.join(DRIVE_DATASET_PATH, 'val/images'), COLAB_VAL_IMAGES, 'validation images')
copy_files(os.path.join(DRIVE_DATASET_PATH, 'val/labels'), COLAB_VAL_LABELS, 'validation labels')

# Copy test data
copy_files(os.path.join(DRIVE_DATASET_PATH, 'test/images'), COLAB_TEST_IMAGES, 'test images')
copy_files(os.path.join(DRIVE_DATASET_PATH, 'test/labels'), COLAB_TEST_LABELS, 'test labels')

print("Dataset copy process complete. Please ensure your ppe.yaml is updated with correct class names and count.")


Copying training images from /content/drive/MyDrive/ppe_yolo11/train/images to /content/drive/MyDrive/ppe/images/train...
Finished copying training images.
Copying training labels from /content/drive/MyDrive/ppe_yolo11/train/labels to /content/drive/MyDrive/ppe/labels/train...
Finished copying training labels.
Copying test images from /content/drive/MyDrive/ppe_yolo11/test/images to /content/drive/MyDrive/ppe/images/test...
Finished copying test images.
Copying test labels from /content/drive/MyDrive/ppe_yolo11/test/labels to /content/drive/MyDrive/ppe/labels/test...
Finished copying test labels.
Dataset copy process complete. Please ensure your ppe.yaml is updated with correct class names and count.


In [ ]:
!find /content/drive/MyDrive -type d -iname "*ppe*" -print0 | xargs -0 -I {} echo "[{}]"

[/content/drive/MyDrive/ppe]
[/content/drive/MyDrive/ppe_yolo11]


In [ ]:
!ls -R "/content/drive/MyDrive/ppe"

/content/drive/MyDrive/ppe:
images	labels	ppe.yaml  test	train  val

/content/drive/MyDrive/ppe/images:
test  train  val

/content/drive/MyDrive/ppe/images/test:

/content/drive/MyDrive/ppe/images/train:

/content/drive/MyDrive/ppe/images/val:

/content/drive/MyDrive/ppe/labels:
test  train  val

/content/drive/MyDrive/ppe/labels/test:

/content/drive/MyDrive/ppe/labels/train:

/content/drive/MyDrive/ppe/labels/val:

/content/drive/MyDrive/ppe/test:
images	labels

/content/drive/MyDrive/ppe/test/images:
 Lab_PPE.png			     'response_0_.output_image (5).jpg'
'response_0_.output_image (10).jpg'  'response_0_.output_image (6).jpg'
'response_0_.output_image (1).jpg'   'response_0_.output_image (7).jpg'
'response_0_.output_image (2).jpg'   'response_0_.output_image (8).jpg'
'response_0_.output_image (3).jpg'   'response_0_.output_image (9).jpg'
'response_0_.output_image (4).jpg'

/content/drive/MyDrive/ppe/test/labels:

/content/drive/MyDrive/ppe/train:
images	labels

/content/drive/MyDrive

In [ ]:
DRIVE_DATASET_PATH = "/content/drive/MyDrive/ppe"

In [ ]:
!ls -R "/content/drive/MyDrive/ppe"

/content/drive/MyDrive/ppe:
images	labels	ppe.yaml  test	train  val

/content/drive/MyDrive/ppe/images:
test  train  val

/content/drive/MyDrive/ppe/images/test:

/content/drive/MyDrive/ppe/images/train:

/content/drive/MyDrive/ppe/images/val:

/content/drive/MyDrive/ppe/labels:
test  train  val

/content/drive/MyDrive/ppe/labels/test:

/content/drive/MyDrive/ppe/labels/train:

/content/drive/MyDrive/ppe/labels/val:

/content/drive/MyDrive/ppe/test:
images	labels

/content/drive/MyDrive/ppe/test/images:
 Lab_PPE.png			     'response_0_.output_image (5).jpg'
'response_0_.output_image (10).jpg'  'response_0_.output_image (6).jpg'
'response_0_.output_image (1).jpg'   'response_0_.output_image (7).jpg'
'response_0_.output_image (2).jpg'   'response_0_.output_image (8).jpg'
'response_0_.output_image (3).jpg'   'response_0_.output_image (9).jpg'
'response_0_.output_image (4).jpg'

/content/drive/MyDrive/ppe/test/labels:

/content/drive/MyDrive/ppe/train:
images	labels

/content/drive/MyDrive

In [125]:
%%writefile /content/drive/MyDrive/ppe/ppe.yaml
train: /content/drive/MyDrive/ppe/train/images
val: /content/drive/MyDrive/ppe/valid/images
test: /content/drive/MyDrive/ppe/test/images

nc: 8
names: ["gloves", "goggles", "lab_coat", "mask", "chemicals", "chemical bottle", "face_shield", "person"]

Overwriting /content/drive/MyDrive/ppe/ppe.yaml


In [ ]:
!find /content/drive -type f -iname "*.zip"

/content/drive/MyDrive/data/input/images/roboflow.zip


In [ ]:
!ls -lh /content/drive/MyDrive/data/input/images/roboflow.zip

-rw------- 1 root root 257 Aug  8 03:40 /content/drive/MyDrive/data/input/images/roboflow.zip


In [ ]:
!head -c 20 /content/drive/MyDrive/data/input/images/roboflow.zip

<?xml version='1.0' 

In [90]:
!find /content/drive -type f -iname "*.zip"

/content/drive/MyDrive/ppe-detection-dataset.v4i.yolov11 (1).zip


In [97]:
!unzip "/content/drive/MyDrive/ppe-detection-dataset.v4i.yolov11 (1).zip" -d /content/drive/MyDrive/ppe_yolo11

Archive:  /content/drive/MyDrive/ppe-detection-dataset.v4i.yolov11 (1).zip
replace /content/drive/MyDrive/ppe_yolo11/test/images/young-hispanic-man-scientist-write-260nw-2193895913_webp.rf.d8c1f25d881276a1b7b125fa4ada541f.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [98]:
!ls -R /content/drive/MyDrive/ppe_yolo11

/content/drive/MyDrive/ppe_yolo11:
data.yaml  test  train	valid

/content/drive/MyDrive/ppe_yolo11/test:
images	labels

/content/drive/MyDrive/ppe_yolo11/test/images:
bottle-bubbles-water-with-blurry-background_664644-216_avif.rf.3beba7e9fb4fadf1a884b293db084676.jpg
female-scientist-wearing-lab-coat-260nw-2661385097_webp.rf.34ab36b2050fdeef4382e7c41c347b14.jpg
fluorescent-solution-ppe-healthcare-workers-donning-doffing_jpg.rf.284fcc321d1aa735aa270989e24a61f8.jpg
front-view-female-doctor-white-medical-suit-sitting-front-table-with-solutions-white-space_140725-84033-removebg-preview-e1675502843449_png.rf.64fc94bb478429d05ddd96f975e8d65b.jpg
images-7-_jpeg.rf.25f1ee172d2e584ee6f2b4fc39a03276.jpg
pexels-artempodrez-5726703_jpg.rf.413f91ff8e3a45708b65ddaccef23709.jpg
pexels-photo-5726701_avif.rf.fafb45385af66d49ee048b03457ddf1a.jpg
videoblocks-female-scientist-with-microscope-in-lab-woman-scientist-doing-microscope-research-microscope-scientist-working-in-lab-lab-scientist-looking-in-micros

In [129]:
!rm -rf /root/.config/Ultralytics

In [130]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")

In [131]:
model.train(
    data="/content/drive/MyDrive/ppe_yolo11/data.yaml",
    epochs=50,
    imgsz=640
)

New https://pypi.org/project/ultralytics/8.4.117 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/ppe_yolo11/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=

Mean of empty slice.
invalid value encountered in divide



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50         0G      1.302      3.634      1.596        121        640: 100% ━━━━━━━━━━━━ 3/3 14.3s/it 42.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 5.0s/it 5.0s
                   all         16          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, cannot compute metrics without labels


Mean of empty slice.
invalid value encountered in divide



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50         0G      1.207      3.521      1.529        117        640: 100% ━━━━━━━━━━━━ 3/3 15.0s/it 45.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 5.8s/it 5.8s
                   all         16          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, cannot compute metrics without labels


...

Mean of empty slice.
invalid value encountered in divide



50 epochs completed in 0.677 hours.
Optimizer stripped from /content/runs/detect/train-13/weights/last.pt, 5.5MB
Optimizer stripped from /content/runs/detect/train-13/weights/best.pt, 5.5MB

Validating /content/runs/detect/train-13/weights/best.pt...
Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO11n summary (fused): 101 layers, 2,583,712 parameters, 0 gradients, 6.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 5.1s/it 5.1s


Mean of empty slice.
invalid value encountered in scalar divide
Mean of empty slice.
invalid value encountered in divide
Mean of empty slice.
invalid value encountered in divide
Mean of empty slice.
invalid value encountered in divide
Mean of empty slice.


                   all         16          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, cannot compute metrics without labels
Speed: 2.6ms preprocess, 241.3ms inference, 0.0ms loss, 67.2ms postprocess per image
Results saved to /content/runs/detect/train-13


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([], dtype=int64)
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e1cc3a55610>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

In [103]:
model.train(
    data="/content/drive/MyDrive/ppe_yolo11/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    lr0=0.001,
    patience=10,
    optimizer="Adam",
    device="cpu"
)

Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/ppe_yolo11/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-10, nbs

Mean of empty slice.
invalid value encountered in divide



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50         0G      1.335      3.418      1.616        121        640: 100% ━━━━━━━━━━━━ 3/3 15.3s/it 46.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 4.5s/it 4.5s
                   all         16          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, cannot compute metrics without labels


Mean of empty slice.
invalid value encountered in divide



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50         0G      1.279       3.17      1.595        117        640: 100% ━━━━━━━━━━━━ 3/3 14.0s/it 41.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 4.5s/it 4.5s
                   all         16          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, cannot compute metrics without labels


...

Mean of empty slice.
invalid value encountered in divide



50 epochs completed in 0.631 hours.
Optimizer stripped from /content/runs/detect/train-10/weights/last.pt, 6.3MB
Optimizer stripped from /content/runs/detect/train-10/weights/best.pt, 6.3MB

Validating /content/runs/detect/train-10/weights/best.pt...
Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Model summary (fused): 73 layers, 3,007,208 parameters, 0 gradients, 8.1 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 4.3s/it 4.3s


Mean of empty slice.
invalid value encountered in scalar divide
Mean of empty slice.
invalid value encountered in divide
Mean of empty slice.
invalid value encountered in divide
Mean of empty slice.
invalid value encountered in divide
Mean of empty slice.


                   all         16          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, cannot compute metrics without labels
Speed: 2.5ms preprocess, 234.3ms inference, 0.0ms loss, 25.7ms postprocess per image
Results saved to /content/runs/detect/train-10


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([], dtype=int64)
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e1cc36ef920>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

In [117]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")
model.train(data="/content/drive/MyDrive/ppe_yolo11/data.yaml")

New https://pypi.org/project/ultralytics/8.4.117 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/ppe_yolo11/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup

Mean of empty slice.
invalid value encountered in divide



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/100         0G      1.301      3.634      1.596        121        640: 100% ━━━━━━━━━━━━ 3/3 14.7s/it 44.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 4.7s/it 4.7s
                   all         16          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, cannot compute metrics without labels


Mean of empty slice.
invalid value encountered in divide



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100         0G       1.21      3.526      1.528        117        640: 100% ━━━━━━━━━━━━ 3/3 14.1s/it 42.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 4.9s/it 4.9s
                   all         16          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, cannot compute metrics without labels


...

Mean of empty slice.
invalid value encountered in divide



100 epochs completed in 1.326 hours.
Optimizer stripped from /content/runs/detect/train-12/weights/last.pt, 5.5MB
Optimizer stripped from /content/runs/detect/train-12/weights/best.pt, 5.5MB

Validating /content/runs/detect/train-12/weights/best.pt...
Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO11n summary (fused): 101 layers, 2,583,712 parameters, 0 gradients, 6.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 5.7s/it 5.7s


Mean of empty slice.
invalid value encountered in scalar divide
Mean of empty slice.
invalid value encountered in divide
Mean of empty slice.
invalid value encountered in divide
Mean of empty slice.
invalid value encountered in divide
Mean of empty slice.


                   all         16          0          0          0          0          0
WARNING ⚠️ no labels found in detect set, cannot compute metrics without labels
Speed: 2.4ms preprocess, 298.6ms inference, 0.0ms loss, 47.6ms postprocess per image
Results saved to /content/runs/detect/train-12


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([], dtype=int64)
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e1cd0b6d100>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

In [124]:
%%writefile /content/drive/MyDrive/ppe_yolo11/data.yaml
train: ../train/images
val: ../valid/images
test: ../test/images

nc: 8
names: ['chemical bottle', 'chemicals', 'face shield', 'gloves', 'googles', 'lab coat', 'mask', 'person']

roboflow:
  workspace: troy-nsofor
  project: ppe-detection-dataset-qdp63
  version: 4
  license: CC BY 4.0
  url: https://universe.roboflow.com/troy-nsofor/ppe-detection-dataset-qdp63/dataset/4

Overwriting /content/drive/MyDrive/ppe_yolo11/data.yaml


In [128]:
!ls -R /content/drive/MyDrive/ppe_yolo11

/content/drive/MyDrive/ppe_yolo11:
data.yaml  test  train	valid

/content/drive/MyDrive/ppe_yolo11/test:
images	labels

/content/drive/MyDrive/ppe_yolo11/test/images:
bottle-bubbles-water-with-blurry-background_664644-216_avif.rf.3beba7e9fb4fadf1a884b293db084676.jpg
female-scientist-wearing-lab-coat-260nw-2661385097_webp.rf.34ab36b2050fdeef4382e7c41c347b14.jpg
fluorescent-solution-ppe-healthcare-workers-donning-doffing_jpg.rf.284fcc321d1aa735aa270989e24a61f8.jpg
front-view-female-doctor-white-medical-suit-sitting-front-table-with-solutions-white-space_140725-84033-removebg-preview-e1675502843449_png.rf.64fc94bb478429d05ddd96f975e8d65b.jpg
images-7-_jpeg.rf.25f1ee172d2e584ee6f2b4fc39a03276.jpg
pexels-artempodrez-5726703_jpg.rf.413f91ff8e3a45708b65ddaccef23709.jpg
pexels-photo-5726701_avif.rf.fafb45385af66d49ee048b03457ddf1a.jpg
videoblocks-female-scientist-with-microscope-in-lab-woman-scientist-doing-microscope-research-microscope-scientist-working-in-lab-lab-scientist-looking-in-micros